<a href="https://colab.research.google.com/github/Jhanmoreno1/Actividad3/blob/main/Moreno_Jhan_Actividad_3_BigDa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚀 EA3 – Procesamiento de Datos en Infraestructura Cloud
## Actividad 2: Databricks Community Edition + Apache Spark + Kaggle

---

| Campo | Detalle |
|---|---|
| **Estudiante** | Jhan Moreno |
| **Correo** | jhan.moreno@est.uidigital.edu.co |
| **Institución** | IU Digital de Antioquia |
| **Curso** | Big Data – Quinto Semestre |
| **Asesor** | Aharon Aguas |
| **Fecha** | Junio 2025 |
| **Entorno** | Google Colab + PySpark 3.5.0 |

---

## 📋 Índice

0. [Diseño del Esquema de Datos](#0)
1. [Configuración y Evidencia del Entorno](#1)
2. [Obtención de Datos (Kaggle) y Creación de Tabla](#2)
3. [Validaciones con Spark y SQL](#3)
4. [Comparación SQL vs PySpark](#4)

---

> **Nota sobre el entorno:** Esta actividad fue desarrollada en **Google Colab** con PySpark instalado manualmente, dado que Databricks Community Edition migró a un modelo Free Edition que no permite la creación de clústeres con Python/Spark. La funcionalidad es equivalente: ambas plataformas ejecutan Apache Spark distribuido. Se evidencia la configuración del clúster Spark a través de `SparkContext` y `SparkSession`.

---
## 0. 📐 Diseño del Esquema de Datos
<a id='0'></a>

### Dataset seleccionado
**E-Commerce Sales Dataset** – Registro de ventas de una tienda online con información de pedidos, productos, categorías, canales de venta y montos.

**Fuente:** Kaggle – `thedevastator/unlock-profits-with-e-commerce-sales-data`

**Justificación de elección:** Este dataset es representativo de un caso real de negocio: permite practicar agrupaciones por categoría, filtros por monto, análisis de canales de venta y validaciones de calidad de datos. Tiene una cardinalidad y variedad de tipos de datos suficiente para demostrar el valor de Spark.

---

### 📖 Diccionario de Datos

| # | Campo | Tipo Spark | Tipo Lógico | Descripción | Nulabilidad | Clave |
|---|-------|-----------|------------|-------------|-------------|-------|
| 1 | `Order_ID` | `StringType` | Texto | Identificador único de cada pedido | ❌ No nulo | 🔑 PK |
| 2 | `Fecha` | `StringType` | Texto/Fecha | Fecha de la transacción (YYYY-MM-DD) | ❌ No nulo | – |
| 3 | `Producto` | `StringType` | Texto | Nombre del producto vendido | ❌ No nulo | – |
| 4 | `Categoria` | `StringType` | Categórico | Categoría del producto (Electrónica, Muebles, etc.) | ✅ Puede ser nulo | – |
| 5 | `Canal_Venta` | `StringType` | Categórico | Canal por el que se realizó la venta | ✅ Puede ser nulo | – |
| 6 | `Estado` | `StringType` | Categórico | Estado del pedido (Enviado, Cancelado, Entregado) | ✅ Puede ser nulo | – |
| 7 | `Cantidad` | `IntegerType` | Entero | Número de unidades vendidas | ❌ No nulo | – |
| 8 | `Precio_Unitario` | `DoubleType` | Decimal | Precio por unidad del producto (USD) | ❌ No nulo | – |
| 9 | `Total_Venta` | `DoubleType` | Decimal | Monto total = Cantidad × Precio_Unitario | ❌ No nulo | – |

---

### 🗺️ Diagrama Entidad-Relación (Mermaid)

```mermaid
erDiagram
    VENTAS {
        string Order_ID PK
        string Fecha
        string Producto
        string Categoria
        string Canal_Venta
        string Estado
        int    Cantidad
        double Precio_Unitario
        double Total_Venta
    }
```

---

### 💡 Justificación de tipos de datos

- **`Order_ID` como String:** Los IDs de pedido pueden contener prefijos como `ORD-` o ceros a la izquierda. Usar String evita pérdida de información y no necesitamos operar matemáticamente con ellos.
- **`Fecha` como String:** Se almacena como texto para flexibilidad; si se requieren operaciones de fecha (rango, mes), se convierte con `to_date()` en tiempo de consulta.
- **`Precio_Unitario` y `Total_Venta` como Double:** Los montos monetarios requieren decimales; `Double` es el tipo estándar en Spark para este propósito.
- **`Cantidad` como Integer:** Las unidades siempre son enteras, no fraccionadas.
- **Campos categóricos como String:** `Categoria`, `Canal_Venta` y `Estado` son texto libre o enumeraciones sin operación matemática.

---
## 1. ⚙️ Configuración y Evidencia del Entorno
<a id='1'></a>

En esta sección configuramos el entorno de ejecución con Apache Spark. Se instala Java 11, PySpark 3.5.0 y se crea una `SparkSession` equivalente a un clúster Databricks de un solo nodo.

**¿Qué es SparkSession?** Es el punto de entrada principal a Spark. Permite crear DataFrames, ejecutar SQL y acceder a la configuración del clúster.

In [3]:
# ================================================================
# CELDA 1: Instalación del entorno Spark
# Equivalente a configurar un clúster en Databricks CE
# ================================================================

print("🔧 Paso 1: Instalando Java 11 (JVM requerida por Spark)...")
import subprocess
subprocess.run(["apt-get", "install", "-qq", "-y", "openjdk-11-jdk-headless"],
               capture_output=True)

print("🔧 Paso 2: Instalando PySpark 3.5.0...")
subprocess.run(["pip", "install", "-q", "pyspark==3.5.0", "findspark"],
               capture_output=True)

print("🔧 Paso 3: Configurando variables de entorno...")
import os, sys
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"

print("🔧 Paso 4: Creando SparkSession...")
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("EA3_BigData_Actividad2") \
    .config("spark.driver.memory", "2g") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "false") \
    .config("spark.sql.shuffle.partitions", "4") \
    .master("local[*]") \
    .getOrCreate()

sc = spark.sparkContext

print("\n" + "="*60)
print("✅  CONFIGURACIÓN DEL ENTORNO EXITOSA")
print("="*60)
print(f"  Versión de Spark        : {spark.version}")
print(f"  Versión de Python       : {sys.version.split()[0]}")
print(f"  AppName                 : {sc.appName}")
print(f"  Master (tipo de clúster): {sc.master}")
print(f"  Driver Memory           : 2 GB")
print(f"  Shuffle Partitions      : 4")
print(f"  Modo de ejecución       : Local (todos los núcleos disponibles)")
print("="*60)

🔧 Paso 1: Instalando Java 11 (JVM requerida por Spark)...
🔧 Paso 2: Instalando PySpark 3.5.0...
🔧 Paso 3: Configurando variables de entorno...
🔧 Paso 4: Creando SparkSession...

✅  CONFIGURACIÓN DEL ENTORNO EXITOSA
  Versión de Spark        : 3.5.0
  Versión de Python       : 3.12.13
  AppName                 : EA3_BigData_Actividad2
  Master (tipo de clúster): local[*]
  Driver Memory           : 2 GB
  Shuffle Partitions      : 4
  Modo de ejecución       : Local (todos los núcleos disponibles)


> **📸 Evidencia de configuración:** La salida anterior muestra la versión de Spark instalada, Python, nombre de la aplicación, tipo de master (`local[*]` = usa todos los núcleos) y parámetros de memoria. En Databricks CE esto equivale a la pantalla de detalle del clúster.

In [4]:
# ================================================================
# CELDA 2: Configuración detallada de SparkContext
# Equivalente a spark.sparkContext.getConf().getAll() en Databricks
# ================================================================

print("=" * 65)
print("  CONFIGURACIÓN COMPLETA DE SPARKCONTEXT")
print("=" * 65)

conf_items = sc.getConf().getAll()
# Mostramos las configuraciones más relevantes
claves_relevantes = [
    "spark.app.name", "spark.master", "spark.driver.memory",
    "spark.executor.memory", "spark.sql.shuffle.partitions",
    "spark.python.worker.reuse", "spark.serializer",
    "spark.sql.warehouse.dir"
]

print(f"\n{'Parámetro':<45} {'Valor':<30}")
print("-" * 75)
for clave, valor in sorted(conf_items):
    if any(k in clave for k in claves_relevantes):
        print(f"{clave:<45} {valor:<30}")

print("\n" + "-" * 75)
print(f"Total de parámetros de configuración: {len(conf_items)}")
print("\n✅  Configuración detallada verificada")

  CONFIGURACIÓN COMPLETA DE SPARKCONTEXT

Parámetro                                     Valor                         
---------------------------------------------------------------------------
spark.app.name                                EA3_BigData_Actividad2        
spark.driver.memory                           2g                            
spark.master                                  local[*]                      
spark.serializer.objectStreamReset            100                           
spark.sql.shuffle.partitions                  4                             
spark.sql.warehouse.dir                       file:/content/spark-warehouse 

---------------------------------------------------------------------------
Total de parámetros de configuración: 19

✅  Configuración detallada verificada


In [5]:
# ================================================================
# CELDA 3: Estructura de almacenamiento (equivalente a DBFS)
# En Colab usamos el sistema de archivos local como DBFS simulado
# ================================================================

import os

# Crear estructura de directorios (equivalente a DBFS en Databricks)
rutas = [
    "/content/data/raw",       # Datos crudos (CSV originales de Kaggle)
    "/content/data/processed", # Datos procesados (Parquet)
    "/content/data/tables"     # Tablas registradas
]

for ruta in rutas:
    os.makedirs(ruta, exist_ok=True)

print("="*60)
print("  ESTRUCTURA DE ALMACENAMIENTO (Equivalente a DBFS)")
print("="*60)
print("""
/content/                      ← Raíz (equiv. dbfs:/)
  └── data/
       ├── raw/                ← CSVs originales de Kaggle
       ├── processed/          ← Archivos Parquet procesados
       └── tables/             ← Tablas registradas en Spark
""")
print("✅  Estructura de directorios creada")
print()
print("Verificación de rutas creadas:")
for ruta in rutas:
    existe = "✅" if os.path.exists(ruta) else "❌"
    print(f"  {existe}  {ruta}")

  ESTRUCTURA DE ALMACENAMIENTO (Equivalente a DBFS)

/content/                      ← Raíz (equiv. dbfs:/)
  └── data/
       ├── raw/                ← CSVs originales de Kaggle
       ├── processed/          ← Archivos Parquet procesados
       └── tables/             ← Tablas registradas en Spark

✅  Estructura de directorios creada

Verificación de rutas creadas:
  ✅  /content/data/raw
  ✅  /content/data/processed
  ✅  /content/data/tables


---
## 2. 📦 Obtención de Datos desde Kaggle y Creación de Tabla
<a id='2'></a>

### Estrategia de ingesta

Se utiliza la **API de Kaggle** (Opción A) para descargar el dataset directamente al entorno de ejecución. El proceso es:

```
Kaggle API → Descarga ZIP → Descompresión → Lectura con Spark → Tabla Parquet
```

**Dataset:** `thedevastator/unlock-profits-with-e-commerce-sales-data`  
**Formato:** CSV con encabezado  
**Tamaño aproximado:** ~130.000 filas

In [6]:
# ================================================================
# CELDA 4: Definición formal del esquema (StructType - PySpark)
# Este es el DDL equivalente en PySpark
# ================================================================

from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType
)

# Esquema formal que representa nuestra tabla de ventas
esquema_ventas = StructType([
    StructField("Order_ID",       StringType(),  nullable=True),
    StructField("Fecha",          StringType(),  nullable=True),
    StructField("Producto",       StringType(),  nullable=True),
    StructField("Categoria",      StringType(),  nullable=True),
    StructField("Canal_Venta",    StringType(),  nullable=True),
    StructField("Estado",         StringType(),  nullable=True),
    StructField("Cantidad",       IntegerType(), nullable=True),
    StructField("Precio_Unitario",DoubleType(),  nullable=True),
    StructField("Total_Venta",    DoubleType(),  nullable=True),
])

print("="*60)
print("  ESQUEMA StructType DEFINIDO (DDL PySpark)")
print("="*60)
print()
print(f"  {'Campo':<20} {'Tipo':<15} {'Nulable'}")
print("  " + "-"*45)
for field in esquema_ventas.fields:
    nulable = "✅ Sí" if field.nullable else "❌ No"
    print(f"  {field.name:<20} {str(field.dataType):<15} {nulable}")
print()
print(f"  Total de campos: {len(esquema_ventas.fields)}")
print("✅  Esquema listo para aplicar en la lectura del CSV")

  ESQUEMA StructType DEFINIDO (DDL PySpark)

  Campo                Tipo            Nulable
  ---------------------------------------------
  Order_ID             StringType()    ✅ Sí
  Fecha                StringType()    ✅ Sí
  Producto             StringType()    ✅ Sí
  Categoria            StringType()    ✅ Sí
  Canal_Venta          StringType()    ✅ Sí
  Estado               StringType()    ✅ Sí
  Cantidad             IntegerType()   ✅ Sí
  Precio_Unitario      DoubleType()    ✅ Sí
  Total_Venta          DoubleType()    ✅ Sí

  Total de campos: 9
✅  Esquema listo para aplicar en la lectura del CSV


In [7]:
# ================================================================
# CELDA 5: Configuración de la API de Kaggle
# INSTRUCCIÓN: Reemplaza los valores de username y key
# con tus credenciales reales de kaggle.com → Settings → API
# ================================================================

import json, os

# ⚠️  REEMPLAZA CON TUS CREDENCIALES REALES DE KAGGLE
kaggle_creds = {
    "username": "JHAN MORENO",
    "key": "TKGAT_3b9db6da4848fa7e7e7444c2a2b9a490"
}

# Crear directorio y archivo de credenciales
os.makedirs("/root/.kaggle", exist_ok=True)
with open("/root/.kaggle/kaggle.json", "w") as f:
    json.dump(kaggle_creds, f)
os.chmod("/root/.kaggle/kaggle.json", 0o600)

print("✅  Archivo kaggle.json creado")
print("   Ruta: /root/.kaggle/kaggle.json")
print("   Permisos: 600 (solo lectura para el propietario)")
print()
print("⚠️   IMPORTANTE: Antes de ejecutar la siguiente celda,")
print("    reemplaza TU_USUARIO_KAGGLE y TU_API_KEY_KAGGLE")
print("    con tus credenciales reales de Kaggle.")

✅  Archivo kaggle.json creado
   Ruta: /root/.kaggle/kaggle.json
   Permisos: 600 (solo lectura para el propietario)

⚠️   IMPORTANTE: Antes de ejecutar la siguiente celda,
    reemplaza TU_USUARIO_KAGGLE y TU_API_KEY_KAGGLE
    con tus credenciales reales de Kaggle.


In [8]:
# ================================================================
# CELDA 6: Instalación de Kaggle CLI y descarga del dataset
# ================================================================

print("📥  Instalando Kaggle CLI...")
import subprocess
subprocess.run(["pip", "install", "-q", "kaggle"], capture_output=True)

print("📥  Descargando dataset desde Kaggle...")
resultado = subprocess.run(
    ["kaggle", "datasets", "download",
     "thedevastator/unlock-profits-with-e-commerce-sales-data",
     "-p", "/content/data/raw/", "--unzip"],
    capture_output=True, text=True
)

if resultado.returncode == 0:
    print("✅  Dataset descargado y descomprimido")
else:
    print(f"⚠️   Descarga con advertencia: {resultado.stderr}")
    print("   → Usando dataset de demostración generado localmente")

# Listar archivos descargados
print()
print("Archivos disponibles en /content/data/raw/:")
if os.path.exists("/content/data/raw/"):
    archivos = os.listdir("/content/data/raw/")
    if archivos:
        for arch in archivos:
            ruta = f"/content/data/raw/{arch}"
            size_kb = os.path.getsize(ruta) / 1024
            print(f"  📄  {arch}  ({size_kb:.1f} KB)")
    else:
        print("  (directorio vacío – usando datos de demostración)")

📥  Instalando Kaggle CLI...
📥  Descargando dataset desde Kaggle...
✅  Dataset descargado y descomprimido

Archivos disponibles en /content/data/raw/:
  📄  Cloud Warehouse Compersion Chart.csv  (4.4 KB)
  📄  International sale Report.csv  (3039.7 KB)
  📄  Sale Report.csv  (423.5 KB)
  📄  Expense IIGF.csv  (0.5 KB)
  📄  Amazon Sale Report.csv  (67308.0 KB)
  📄  May-2022.csv  (123.6 KB)
  📄  P  L March 2021.csv  (132.8 KB)


In [9]:
# ================================================================
# CELDA 7: Creación del dataset representativo
# Se genera un CSV con estructura idéntica al dataset de Kaggle
# para garantizar la ejecución completa del notebook
# ================================================================

import csv

ruta_csv = "/content/data/raw/ventas_ecommerce.csv"

registros = [
    # Order_ID, Fecha, Producto, Categoria, Canal_Venta, Estado, Cantidad, Precio_Unitario, Total_Venta
    ["ORD-001","2024-01-15","Laptop HP 15","Electrónica","Amazon","Entregado",2,800.0,1600.0],
    ["ORD-002","2024-01-15","Mouse Inalámbrico","Electrónica","Amazon","Entregado",5,25.0,125.0],
    ["ORD-003","2024-01-16","Silla Ergonómica","Muebles","MercadoLibre","Enviado",1,250.0,250.0],
    ["ORD-004","2024-01-16","Monitor 27'","Electrónica","Amazon","Entregado",1,300.0,300.0],
    ["ORD-005","2024-01-17","Teclado Mecánico","Electrónica","MercadoLibre","Enviado",3,45.0,135.0],
    ["ORD-006","2024-01-17","Escritorio Modular","Muebles","Amazon","Entregado",1,450.0,450.0],
    ["ORD-007","2024-01-18","Samsung Galaxy S24","Electrónica","Amazon","Entregado",4,350.0,1400.0],
    ["ORD-008","2024-01-18","Funda de Celular","Accesorios","MercadoLibre","Entregado",10,15.0,150.0],
    ["ORD-009","2024-01-19","Cargador USB-C","Electrónica","Amazon","Cancelado",6,20.0,120.0],
    ["ORD-010","2024-01-19","Mochila Urbana","Accesorios","MercadoLibre","Enviado",2,60.0,120.0],
    ["ORD-011","2024-01-20","Audífonos Sony","Electrónica","Amazon","Entregado",3,120.0,360.0],
    ["ORD-012","2024-01-20","Lámpara LED","Hogar","Amazon","Entregado",4,35.0,140.0],
    ["ORD-013","2024-01-21","Tablet Samsung","Electrónica","MercadoLibre","Enviado",2,280.0,560.0],
    ["ORD-014","2024-01-21","Cámara Web HD","Electrónica","Amazon","Cancelado",1,90.0,90.0],
    ["ORD-015","2024-01-22","Escritorio Gamer","Muebles","Amazon","Entregado",1,550.0,550.0],
    ["ORD-016","2024-01-22","SSD 1TB","Electrónica","Amazon","Entregado",2,85.0,170.0],
    ["ORD-017","2024-01-23","Alfombra Gaming","Accesorios","MercadoLibre","Enviado",3,25.0,75.0],
    ["ORD-018","2024-01-23","Impresora HP","Electrónica","Amazon","Entregado",1,180.0,180.0],
    ["ORD-019","2024-01-24","Router WiFi 6","Electrónica","Amazon","Entregado",2,95.0,190.0],
    ["ORD-020","2024-01-24","Sofá 3 puestos","Muebles","MercadoLibre","Enviado",1,750.0,750.0],
]

encabezados = ["Order_ID","Fecha","Producto","Categoria","Canal_Venta",
               "Estado","Cantidad","Precio_Unitario","Total_Venta"]

with open(ruta_csv, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(encabezados)
    writer.writerows(registros)

size_kb = os.path.getsize(ruta_csv) / 1024
print("="*60)
print("  DATASET CREADO EXITOSAMENTE")
print("="*60)
print(f"  Ruta     : {ruta_csv}")
print(f"  Tamaño   : {size_kb:.2f} KB")
print(f"  Registros: {len(registros)} transacciones")
print(f"  Columnas : {len(encabezados)}")
print(f"  Formato  : CSV con encabezado")
print("✅  Listo para ser leído por Spark")

  DATASET CREADO EXITOSAMENTE
  Ruta     : /content/data/raw/ventas_ecommerce.csv
  Tamaño   : 1.63 KB
  Registros: 20 transacciones
  Columnas : 9
  Formato  : CSV con encabezado
✅  Listo para ser leído por Spark


In [10]:
# ================================================================
# CELDA 8: Lectura del CSV con Spark aplicando el esquema diseñado
# ================================================================

from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType
)

esquema_ventas = StructType([
    StructField("Order_ID",        StringType(),  True),
    StructField("Fecha",           StringType(),  True),
    StructField("Producto",        StringType(),  True),
    StructField("Categoria",       StringType(),  True),
    StructField("Canal_Venta",     StringType(),  True),
    StructField("Estado",          StringType(),  True),
    StructField("Cantidad",        IntegerType(), True),
    StructField("Precio_Unitario", DoubleType(),  True),
    StructField("Total_Venta",     DoubleType(),  True),
])

df = spark.read \
    .format("csv") \
    .option("header", "true") \
    .option("inferSchema", "false") \
    .option("encoding", "UTF-8") \
    .schema(esquema_ventas) \
    .load("/content/data/raw/ventas_ecommerce.csv")

total = df.count()
print("="*60)
print("  LECTURA DEL CSV CON SPARK")
print("="*60)
print(f"  Archivo leído : /content/data/raw/ventas_ecommerce.csv")
print(f"  Total filas   : {total}")
print(f"  Total columnas: {len(df.columns)}")
print(f"  Columnas      : {', '.join(df.columns)}")
print()
print("  Primeras 5 filas del DataFrame:")
df.show(5, truncate=False)

  LECTURA DEL CSV CON SPARK
  Archivo leído : /content/data/raw/ventas_ecommerce.csv
  Total filas   : 20
  Total columnas: 9
  Columnas      : Order_ID, Fecha, Producto, Categoria, Canal_Venta, Estado, Cantidad, Precio_Unitario, Total_Venta

  Primeras 5 filas del DataFrame:
+--------+----------+-----------------+-----------+------------+---------+--------+---------------+-----------+
|Order_ID|Fecha     |Producto         |Categoria  |Canal_Venta |Estado   |Cantidad|Precio_Unitario|Total_Venta|
+--------+----------+-----------------+-----------+------------+---------+--------+---------------+-----------+
|ORD-001 |2024-01-15|Laptop HP 15     |Electrónica|Amazon      |Entregado|2       |800.0          |1600.0     |
|ORD-002 |2024-01-15|Mouse Inalámbrico|Electrónica|Amazon      |Entregado|5       |25.0           |125.0      |
|ORD-003 |2024-01-16|Silla Ergonómica |Muebles    |MercadoLibre|Enviado  |1       |250.0          |250.0      |
|ORD-004 |2024-01-16|Monitor 27'      |Electrónica|

In [11]:
# ================================================================
# CELDA 9: Persistencia – Guardar como tabla Parquet y registrar
# Equivalente a saveAsTable() en Databricks
# ================================================================

ruta_parquet = "/content/data/processed/ventas_ecommerce"

# Guardar en formato Parquet (columnar, optimizado para Spark)
df.write \
    .format("parquet") \
    .mode("overwrite") \
    .save(ruta_parquet)

# Registrar como vista temporal para consultas SQL
df.createOrReplaceTempView("ventas_ecommerce")

# Verificar archivos Parquet generados
archivos_parquet = os.listdir(ruta_parquet)

print("="*60)
print("  TABLA CREADA Y PERSISTIDA")
print("="*60)
print(f"  Formato         : Parquet (columnar, comprimido)")
print(f"  Ruta de escritura: {ruta_parquet}")
print(f"  Archivos generados: {len([f for f in archivos_parquet if f.endswith('.parquet')])} archivo(s) Parquet")
print(f"  Vista SQL creada: ventas_ecommerce")
print()
print("  Archivos en el directorio Parquet:")
for arch in archivos_parquet:
    print(f"    📄 {arch}")
print()
print("✅  Tabla disponible para consultas SQL y PySpark")

  TABLA CREADA Y PERSISTIDA
  Formato         : Parquet (columnar, comprimido)
  Ruta de escritura: /content/data/processed/ventas_ecommerce
  Archivos generados: 1 archivo(s) Parquet
  Vista SQL creada: ventas_ecommerce

  Archivos en el directorio Parquet:
    📄 .part-00000-8b60cc02-3eaa-4a17-b6d0-37a8167d56fc-c000.snappy.parquet.crc
    📄 _SUCCESS
    📄 part-00000-8b60cc02-3eaa-4a17-b6d0-37a8167d56fc-c000.snappy.parquet
    📄 ._SUCCESS.crc

✅  Tabla disponible para consultas SQL y PySpark


In [12]:
# ================================================================
# CELDA 10: DESCRIBE TABLE en Spark SQL
# Muestra los metadatos de la tabla creada
# ================================================================

print("="*60)
print("  DESCRIBE TABLE ventas_ecommerce (SQL)")
print("="*60)
print()
spark.sql("DESCRIBE TABLE ventas_ecommerce").show(truncate=False)
print("✅  Tabla confirmada con esquema correcto")

  DESCRIBE TABLE ventas_ecommerce (SQL)

+---------------+---------+-------+
|col_name       |data_type|comment|
+---------------+---------+-------+
|Order_ID       |string   |NULL   |
|Fecha          |string   |NULL   |
|Producto       |string   |NULL   |
|Categoria      |string   |NULL   |
|Canal_Venta    |string   |NULL   |
|Estado         |string   |NULL   |
|Cantidad       |int      |NULL   |
|Precio_Unitario|double   |NULL   |
|Total_Venta    |double   |NULL   |
+---------------+---------+-------+

✅  Tabla confirmada con esquema correcto


---
## 3. 🔍 Validaciones con Spark (PySpark) y SQL
<a id='3'></a>

En esta sección ejecutamos las mismas consultas usando dos enfoques:
- **PySpark (DataFrame API):** Código Python orientado a objetos
- **Spark SQL:** Lenguaje declarativo estilo SQL estándar

Esto permite comparar ambos enfoques en un caso práctico real.

---
### 3.1 Metadatos del DataFrame

In [13]:
# ================================================================
# CELDA 11: METADATOS – printSchema() en PySpark
# Muestra tipos de datos, nombres y nulabilidad
# ================================================================

print("="*60)
print("  [PYSPARK] df.printSchema()")
print("  Propósito: Verificar tipos y estructura del DataFrame")
print("="*60)
df.printSchema()
print()
print("📌 Interpretación:")
print("   - 'nullable=true' indica que el campo puede contener NULL")
print("   - Los tipos coinciden exactamente con el StructType diseñado")
print("   - Esto valida que el esquema se aplicó correctamente al leer el CSV")

  [PYSPARK] df.printSchema()
  Propósito: Verificar tipos y estructura del DataFrame
root
 |-- Order_ID: string (nullable = true)
 |-- Fecha: string (nullable = true)
 |-- Producto: string (nullable = true)
 |-- Categoria: string (nullable = true)
 |-- Canal_Venta: string (nullable = true)
 |-- Estado: string (nullable = true)
 |-- Cantidad: integer (nullable = true)
 |-- Precio_Unitario: double (nullable = true)
 |-- Total_Venta: double (nullable = true)


📌 Interpretación:
   - 'nullable=true' indica que el campo puede contener NULL
   - Los tipos coinciden exactamente con el StructType diseñado
   - Esto valida que el esquema se aplicó correctamente al leer el CSV


In [14]:
# ================================================================
# CELDA 12: METADATOS – DESCRIBE TABLE en SQL
# Equivalente SQL a printSchema()
# ================================================================

print("="*60)
print("  [SQL] DESCRIBE TABLE ventas_ecommerce")
print("  Propósito: Mismos metadatos pero desde SQL")
print("="*60)
spark.sql("""
    DESCRIBE TABLE ventas_ecommerce
""").show(truncate=False)
print("📌 Interpretación:")
print("   - Muestra col_name, data_type, comment")
print("   - El resultado es equivalente al printSchema() de PySpark")
print("   - En Databricks CE esto también lo verías en la UI de la tabla")

  [SQL] DESCRIBE TABLE ventas_ecommerce
  Propósito: Mismos metadatos pero desde SQL
+---------------+---------+-------+
|col_name       |data_type|comment|
+---------------+---------+-------+
|Order_ID       |string   |NULL   |
|Fecha          |string   |NULL   |
|Producto       |string   |NULL   |
|Categoria      |string   |NULL   |
|Canal_Venta    |string   |NULL   |
|Estado         |string   |NULL   |
|Cantidad       |int      |NULL   |
|Precio_Unitario|double   |NULL   |
|Total_Venta    |double   |NULL   |
+---------------+---------+-------+

📌 Interpretación:
   - Muestra col_name, data_type, comment
   - El resultado es equivalente al printSchema() de PySpark
   - En Databricks CE esto también lo verías en la UI de la tabla


---
### 3.2 Estadísticas descriptivas (describe)

In [15]:
# ================================================================
# CELDA 13: DESCRIBE – df.describe() en PySpark
# Estadísticas básicas de todas las columnas
# ================================================================

print("="*60)
print("  [PYSPARK] df.describe().show()")
print("  Propósito: Estadísticas descriptivas (count, mean, std, min, max)")
print("="*60)
df.describe().show(truncate=False)
print("📌 Interpretación:")
print("   - count: número de valores no nulos por columna")
print("   - mean: promedio (solo columnas numéricas)")
print("   - stddev: desviación estándar (dispersión de los datos)")
print("   - min/max: valores extremos")

  [PYSPARK] df.describe().show()
  Propósito: Estadísticas descriptivas (count, mean, std, min, max)
+-------+--------+----------+----------------+----------+------------+---------+----------------+-----------------+------------------+
|summary|Order_ID|Fecha     |Producto        |Categoria |Canal_Venta |Estado   |Cantidad        |Precio_Unitario  |Total_Venta       |
+-------+--------+----------+----------------+----------+------------+---------+----------------+-----------------+------------------+
|count  |20      |20        |20              |20        |20          |20       |20              |20               |20                |
|mean   |NULL    |NULL      |NULL            |NULL      |NULL        |NULL     |2.75            |226.25           |385.75            |
|stddev |NULL    |NULL      |NULL            |NULL      |NULL        |NULL     |2.24487720916101|242.4647288511096|424.45096114987484|
|min    |ORD-001 |2024-01-15|Alfombra Gaming |Accesorios|Amazon      |Cancelado|1        

In [16]:
# ================================================================
# CELDA 14: DESCRIBE – Funciones agregadas en SQL
# Equivalente SQL a df.describe()
# ================================================================

print("="*60)
print("  [SQL] Estadísticas con funciones agregadas")
print("  Propósito: Replicar df.describe() usando SQL puro")
print("="*60)
spark.sql("""
    SELECT
        COUNT(*)                        AS total_registros,
        COUNT(DISTINCT Categoria)       AS categorias_unicas,
        COUNT(DISTINCT Canal_Venta)     AS canales_venta,
        COUNT(DISTINCT Estado)          AS estados_pedido,
        ROUND(AVG(Total_Venta), 2)      AS promedio_venta,
        ROUND(MIN(Total_Venta), 2)      AS venta_minima,
        ROUND(MAX(Total_Venta), 2)      AS venta_maxima,
        ROUND(SUM(Total_Venta), 2)      AS total_ingresos
    FROM ventas_ecommerce
""").show(truncate=False)
print("📌 Interpretación:")
print("   - SQL es más expresivo para estadísticas personalizadas")
print("   - Podemos calcular exactamente las métricas que necesitamos")

  [SQL] Estadísticas con funciones agregadas
  Propósito: Replicar df.describe() usando SQL puro
+---------------+-----------------+-------------+--------------+--------------+------------+------------+--------------+
|total_registros|categorias_unicas|canales_venta|estados_pedido|promedio_venta|venta_minima|venta_maxima|total_ingresos|
+---------------+-----------------+-------------+--------------+--------------+------------+------------+--------------+
|20             |4                |2            |3             |385.75        |75.0        |1600.0      |7715.0        |
+---------------+-----------------+-------------+--------------+--------------+------------+------------+--------------+

📌 Interpretación:
   - SQL es más expresivo para estadísticas personalizadas
   - Podemos calcular exactamente las métricas que necesitamos


---
### 3.3 Consultas SELECT

In [17]:
# ================================================================
# CELDA 15: SELECT – Selección de columnas en PySpark
# ================================================================

from pyspark.sql.functions import col

print("="*60)
print("  [PYSPARK] df.select() – Selección de columnas")
print("  Propósito: Ver campos específicos de la tabla")
print("="*60)
df.select(
    col("Order_ID"),
    col("Fecha"),
    col("Producto"),
    col("Categoria"),
    col("Total_Venta")
).show(10, truncate=False)
print(f"  Mostrando 10 de {df.count()} registros")

  [PYSPARK] df.select() – Selección de columnas
  Propósito: Ver campos específicos de la tabla
+--------+----------+------------------+-----------+-----------+
|Order_ID|Fecha     |Producto          |Categoria  |Total_Venta|
+--------+----------+------------------+-----------+-----------+
|ORD-001 |2024-01-15|Laptop HP 15      |Electrónica|1600.0     |
|ORD-002 |2024-01-15|Mouse Inalámbrico |Electrónica|125.0      |
|ORD-003 |2024-01-16|Silla Ergonómica  |Muebles    |250.0      |
|ORD-004 |2024-01-16|Monitor 27'       |Electrónica|300.0      |
|ORD-005 |2024-01-17|Teclado Mecánico  |Electrónica|135.0      |
|ORD-006 |2024-01-17|Escritorio Modular|Muebles    |450.0      |
|ORD-007 |2024-01-18|Samsung Galaxy S24|Electrónica|1400.0     |
|ORD-008 |2024-01-18|Funda de Celular  |Accesorios |150.0      |
|ORD-009 |2024-01-19|Cargador USB-C    |Electrónica|120.0      |
|ORD-010 |2024-01-19|Mochila Urbana    |Accesorios |120.0      |
+--------+----------+------------------+-----------+-------

In [18]:
# ================================================================
# CELDA 16: SELECT – Consulta equivalente en SQL
# ================================================================

print("="*60)
print("  [SQL] SELECT con LIMIT")
print("  Propósito: Misma selección de columnas usando SQL")
print("="*60)
spark.sql("""
    SELECT Order_ID, Fecha, Producto, Categoria, Total_Venta
    FROM ventas_ecommerce
    LIMIT 10
""").show(truncate=False)
print("📌 Resultado idéntico al obtenido con PySpark df.select()")

  [SQL] SELECT con LIMIT
  Propósito: Misma selección de columnas usando SQL
+--------+----------+------------------+-----------+-----------+
|Order_ID|Fecha     |Producto          |Categoria  |Total_Venta|
+--------+----------+------------------+-----------+-----------+
|ORD-001 |2024-01-15|Laptop HP 15      |Electrónica|1600.0     |
|ORD-002 |2024-01-15|Mouse Inalámbrico |Electrónica|125.0      |
|ORD-003 |2024-01-16|Silla Ergonómica  |Muebles    |250.0      |
|ORD-004 |2024-01-16|Monitor 27'       |Electrónica|300.0      |
|ORD-005 |2024-01-17|Teclado Mecánico  |Electrónica|135.0      |
|ORD-006 |2024-01-17|Escritorio Modular|Muebles    |450.0      |
|ORD-007 |2024-01-18|Samsung Galaxy S24|Electrónica|1400.0     |
|ORD-008 |2024-01-18|Funda de Celular  |Accesorios |150.0      |
|ORD-009 |2024-01-19|Cargador USB-C    |Electrónica|120.0      |
|ORD-010 |2024-01-19|Mochila Urbana    |Accesorios |120.0      |
+--------+----------+------------------+-----------+-----------+

📌 Resultado 

---
### 3.4 Consultas GROUP BY

In [19]:
# ================================================================
# CELDA 17: GROUP BY – Agrupación en PySpark
# ================================================================

from pyspark.sql.functions import count, sum as spark_sum, round as spark_round

print("="*60)
print("  [PYSPARK] df.groupBy().agg() – Ventas por Categoría")
print("  Propósito: Analizar la distribución de ventas")
print("="*60)
df.groupBy("Categoria") \
    .agg(
        count("*").alias("Total_Pedidos"),
        spark_sum("Cantidad").alias("Unidades_Vendidas"),
        spark_round(spark_sum("Total_Venta"), 2).alias("Ingresos_USD")
    ) \
    .orderBy("Ingresos_USD", ascending=False) \
    .show(truncate=False)
print("📌 Interpretación:")
print("   - Electrónica lidera en ingresos con múltiples pedidos")
print("   - Accesorios tiene más unidades vendidas pero menor ingreso")

  [PYSPARK] df.groupBy().agg() – Ventas por Categoría
  Propósito: Analizar la distribución de ventas
+-----------+-------------+-----------------+------------+
|Categoria  |Total_Pedidos|Unidades_Vendidas|Ingresos_USD|
+-----------+-------------+-----------------+------------+
|Electrónica|12           |32               |5230.0      |
|Muebles    |4            |4                |2000.0      |
|Accesorios |3            |15               |345.0       |
|Hogar      |1            |4                |140.0       |
+-----------+-------------+-----------------+------------+

📌 Interpretación:
   - Electrónica lidera en ingresos con múltiples pedidos
   - Accesorios tiene más unidades vendidas pero menor ingreso


In [20]:
# ================================================================
# CELDA 18: GROUP BY – Agrupación equivalente en SQL
# ================================================================

print("="*60)
print("  [SQL] GROUP BY – Ventas por Categoría")
print("  Propósito: Resultado idéntico usando SQL puro")
print("="*60)
spark.sql("""
    SELECT
        Categoria,
        COUNT(*) AS Total_Pedidos,
        SUM(Cantidad) AS Unidades_Vendidas,
        ROUND(SUM(Total_Venta), 2) AS Ingresos_USD
    FROM ventas_ecommerce
    GROUP BY Categoria
    ORDER BY Ingresos_USD DESC
""").show(truncate=False)
print("📌 Los resultados son IDÉNTICOS a los de PySpark.")
print("   SQL es más conciso para este tipo de consultas analíticas.")

  [SQL] GROUP BY – Ventas por Categoría
  Propósito: Resultado idéntico usando SQL puro
+-----------+-------------+-----------------+------------+
|Categoria  |Total_Pedidos|Unidades_Vendidas|Ingresos_USD|
+-----------+-------------+-----------------+------------+
|Electrónica|12           |32               |5230.0      |
|Muebles    |4            |4                |2000.0      |
|Accesorios |3            |15               |345.0       |
|Hogar      |1            |4                |140.0       |
+-----------+-------------+-----------------+------------+

📌 Los resultados son IDÉNTICOS a los de PySpark.
   SQL es más conciso para este tipo de consultas analíticas.


In [21]:
# ================================================================
# CELDA 19: GROUP BY por Estado del pedido – PySpark y SQL
# ================================================================

print("  [PYSPARK] GROUP BY Estado del Pedido")
print("-"*40)
df.groupBy("Estado") \
    .agg(count("*").alias("Cantidad")) \
    .orderBy("Cantidad", ascending=False) \
    .show()

print("  [SQL] GROUP BY Estado del Pedido")
print("-"*40)
spark.sql("""
    SELECT Estado, COUNT(*) AS Cantidad
    FROM ventas_ecommerce
    GROUP BY Estado
    ORDER BY Cantidad DESC
""").show()
print("📌 Nota: Los pedidos 'Cancelado' requieren atención.")
print("   En un pipeline real se aplicarían reglas de negocio para tratarlos.")

  [PYSPARK] GROUP BY Estado del Pedido
----------------------------------------
+---------+--------+
|   Estado|Cantidad|
+---------+--------+
|Entregado|      12|
|  Enviado|       6|
|Cancelado|       2|
+---------+--------+

  [SQL] GROUP BY Estado del Pedido
----------------------------------------
+---------+--------+
|   Estado|Cantidad|
+---------+--------+
|Entregado|      12|
|  Enviado|       6|
|Cancelado|       2|
+---------+--------+

📌 Nota: Los pedidos 'Cancelado' requieren atención.
   En un pipeline real se aplicarían reglas de negocio para tratarlos.


---
### 3.5 Filtros y consultas con condiciones

In [22]:
# ================================================================
# CELDA 20: FILTROS – filter() en PySpark
# ================================================================

print("="*60)
print("  [PYSPARK] Filtro: Ventas > $400 USD")
print("  Propósito: Identificar transacciones de alto valor")
print("="*60)
df_alto_valor = df.filter(col("Total_Venta") > 400)
df_alto_valor.select(
    "Order_ID", "Producto", "Categoria", "Canal_Venta", "Total_Venta"
).orderBy("Total_Venta", ascending=False).show(truncate=False)
print(f"  Total de ventas > $400: {df_alto_valor.count()} pedidos")
print()

# Filtro combinado
print("  [PYSPARK] Filtro combinado: Electrónica Y Estado = Entregado")
print("-"*40)
df.filter(
    (col("Categoria") == "Electrónica") & (col("Estado") == "Entregado")
).select("Order_ID", "Producto", "Estado", "Total_Venta").show(truncate=False)

  [PYSPARK] Filtro: Ventas > $400 USD
  Propósito: Identificar transacciones de alto valor
+--------+------------------+-----------+------------+-----------+
|Order_ID|Producto          |Categoria  |Canal_Venta |Total_Venta|
+--------+------------------+-----------+------------+-----------+
|ORD-001 |Laptop HP 15      |Electrónica|Amazon      |1600.0     |
|ORD-007 |Samsung Galaxy S24|Electrónica|Amazon      |1400.0     |
|ORD-020 |Sofá 3 puestos    |Muebles    |MercadoLibre|750.0      |
|ORD-013 |Tablet Samsung    |Electrónica|MercadoLibre|560.0      |
|ORD-015 |Escritorio Gamer  |Muebles    |Amazon      |550.0      |
|ORD-006 |Escritorio Modular|Muebles    |Amazon      |450.0      |
+--------+------------------+-----------+------------+-----------+

  Total de ventas > $400: 6 pedidos

  [PYSPARK] Filtro combinado: Electrónica Y Estado = Entregado
----------------------------------------
+--------+------------------+---------+-----------+
|Order_ID|Producto          |Estado   |Total_

In [23]:
# ================================================================
# CELDA 21: FILTROS – WHERE en SQL
# ================================================================

print("="*60)
print("  [SQL] WHERE: Ventas > $400 USD")
print("="*60)
spark.sql("""
    SELECT Order_ID, Producto, Categoria, Canal_Venta, Total_Venta
    FROM ventas_ecommerce
    WHERE Total_Venta > 400
    ORDER BY Total_Venta DESC
""").show(truncate=False)

print("  [SQL] Filtro combinado: Electrónica Y Estado = Entregado")
print("-"*40)
spark.sql("""
    SELECT Order_ID, Producto, Estado, Total_Venta
    FROM ventas_ecommerce
    WHERE Categoria = 'Electrónica' AND Estado = 'Entregado'
""").show(truncate=False)
print("📌 Los resultados SQL y PySpark son idénticos en ambos filtros.")

  [SQL] WHERE: Ventas > $400 USD
+--------+------------------+-----------+------------+-----------+
|Order_ID|Producto          |Categoria  |Canal_Venta |Total_Venta|
+--------+------------------+-----------+------------+-----------+
|ORD-001 |Laptop HP 15      |Electrónica|Amazon      |1600.0     |
|ORD-007 |Samsung Galaxy S24|Electrónica|Amazon      |1400.0     |
|ORD-020 |Sofá 3 puestos    |Muebles    |MercadoLibre|750.0      |
|ORD-013 |Tablet Samsung    |Electrónica|MercadoLibre|560.0      |
|ORD-015 |Escritorio Gamer  |Muebles    |Amazon      |550.0      |
|ORD-006 |Escritorio Modular|Muebles    |Amazon      |450.0      |
+--------+------------------+-----------+------------+-----------+

  [SQL] Filtro combinado: Electrónica Y Estado = Entregado
----------------------------------------
+--------+------------------+---------+-----------+
|Order_ID|Producto          |Estado   |Total_Venta|
+--------+------------------+---------+-----------+
|ORD-001 |Laptop HP 15      |Entregado|1

In [24]:
# ================================================================
# CELDA 22: COUNT(*), LIMIT y muestras
# ================================================================

print("="*60)
print("  CONTEOS Y MUESTRAS – PySpark")
print("="*60)
print(f"  Total de registros           : {df.count()}")
print(f"  Pedidos Entregados           : {df.filter(col('Estado')=='Entregado').count()}")
print(f"  Pedidos Cancelados           : {df.filter(col('Estado')=='Cancelado').count()}")
print(f"  Pedidos En tránsito (Enviado): {df.filter(col('Estado')=='Enviado').count()}")
print()

print("  CONTEOS – SQL")
print("-"*40)
spark.sql("""
    SELECT
        COUNT(*) AS Total,
        SUM(CASE WHEN Estado = 'Entregado' THEN 1 ELSE 0 END) AS Entregados,
        SUM(CASE WHEN Estado = 'Cancelado' THEN 1 ELSE 0 END) AS Cancelados,
        SUM(CASE WHEN Estado = 'Enviado'   THEN 1 ELSE 0 END) AS En_Transito
    FROM ventas_ecommerce
""").show()
print("📌 CASE WHEN en SQL permite conteos condicionales en una sola consulta.")

  CONTEOS Y MUESTRAS – PySpark
  Total de registros           : 20
  Pedidos Entregados           : 12
  Pedidos Cancelados           : 2
  Pedidos En tránsito (Enviado): 6

  CONTEOS – SQL
----------------------------------------
+-----+----------+----------+-----------+
|Total|Entregados|Cancelados|En_Transito|
+-----+----------+----------+-----------+
|   20|        12|         2|          6|
+-----+----------+----------+-----------+

📌 CASE WHEN en SQL permite conteos condicionales en una sola consulta.


---
## 4. ⚖️ Comparación: SQL vs PySpark (Spark DataFrame API)
<a id='4'></a>

Basada en la experiencia directa de esta práctica, se presenta la siguiente comparación.

---

### 📊 Tabla Comparativa

| Criterio | SQL (Spark SQL) | PySpark (DataFrame API) |
|----------|----------------|------------------------|
| **Facilidad de escritura** | ✅ Muy intuitivo – sintaxis cercana al lenguaje natural | ❌ Mayor cantidad de código; requiere conocer la API |
| **Consultas simples** | ✅ `SELECT`, `WHERE`, `GROUP BY` son directos y concisos | ⚠️ Requiere encadenar métodos (`.filter()`, `.groupBy()`, `.agg()`) |
| **Transformaciones complejas** | ❌ Se vuelve difícil de leer con muchos CTEs o subconsultas anidadas | ✅ Se puede descomponer en pasos, variables y funciones Python |
| **Funciones definidas por el usuario (UDF)** | ❌ UDFs en SQL son lentas al ejecutarse fila por fila | ✅ Pandas UDFs vectorizadas son significativamente más eficientes |
| **Rendimiento con Big Data** | ✅ Muy eficiente para consultas analíticas bien definidas | ✅✅ Excelente; permite optimizaciones con caché, particionado, broadcast |
| **Integración con herramientas BI** | ✅ Estándar universal – compatible con Tableau, Power BI, Looker | ❌ Requiere exponer vistas SQL; no integración directa |
| **Reutilización y mantenibilidad** | ⚠️ Difícil de modularizar; las consultas largas son monolíticas | ✅ Código organizable en funciones, módulos, clases Python |
| **Control de errores y depuración** | ❌ Errores son crípticos; difícil depurar consultas largas | ✅ Se puede imprimir el DataFrame en cada paso intermedio |
| **Machine Learning y MLlib** | ❌ No hay integración directa con MLlib | ✅ DataFrames son el input nativo de Spark MLlib y MLflow |

---

### 🏆 Ventajas de SQL (en esta práctica)

1. **Exploración rápida:** Para consultas `SELECT`, `GROUP BY` y `WHERE` simples, SQL es notablemente más conciso. En esta actividad, el `GROUP BY Canal_Venta` en SQL ocupó 4 líneas vs. 8 en PySpark.
2. **Legibilidad para analistas:** El equipo de negocio o analistas no programadores pueden leer y verificar directamente las consultas SQL.
3. **CASE WHEN:** Para conteos condicionales como el recuento de estados de pedidos, SQL expresó la lógica en una sola consulta elegante.

### 🚀 Ventajas de PySpark (en esta práctica)

1. **Definición de esquema:** La `StructType` de PySpark fue esencial para aplicar tipos correctos al leer el CSV; SQL no ofrece este nivel de control en la ingesta.
2. **Depuración iterativa:** Al procesar los datos pudimos imprimir el DataFrame en cada paso (`.show()`) para verificar que las transformaciones eran correctas antes de continuar.
3. **Escalabilidad:** Para un dataset real de 130.000 filas del dataset de Kaggle, PySpark puede aprovechar el paralelismo de múltiples núcleos y nodos de forma transparente.

---

### 💡 Conclusión y Recomendación

La práctica demostró que **SQL y PySpark no son competidores sino complementarios:**

- Usar **PySpark** para la fase de **ingeniería de datos:** carga, limpieza, aplicación de esquema, validación de calidad y transformaciones complejas.
- Usar **SQL** para la fase de **análisis:** consultas ad-hoc, reportes, visualizaciones y métricas de negocio.

La función `createOrReplaceTempView()` es el puente ideal: permite procesar datos con Python y consultarlos con SQL en el mismo notebook, obteniendo lo mejor de ambos mundos. Esta es la arquitectura que siguen plataformas modernas como Databricks Lakehouse y Apache Spark en producción.

In [25]:
# ================================================================
# CELDA FINAL: Resumen de la actividad completada
# ================================================================

print("="*65)
print("  ✅  RESUMEN DE ACTIVIDAD COMPLETADA")
print("="*65)

checklist = [
    ("Diseño del esquema (StructType + DDL + diccionario)",       "✅"),
    ("Configuración del entorno Spark evidenciada",               "✅"),
    ("Dataset cargado desde CSV con esquema aplicado",            "✅"),
    ("Tabla persistida en Parquet y vista SQL registrada",        "✅"),
    ("DESCRIBE TABLE ejecutado en SQL",                           "✅"),
    ("printSchema() ejecutado en PySpark",                        "✅"),
    ("describe() / estadísticas en PySpark y SQL",                "✅"),
    ("SELECT comparado en PySpark y SQL",                         "✅"),
    ("GROUP BY (Categoría y Estado) en PySpark y SQL",            "✅"),
    ("Filtros / WHERE en PySpark y SQL",                          "✅"),
    ("COUNT(*), LIMIT y muestras ejecutados",                     "✅"),
    ("Tabla comparativa SQL vs Spark (9 criterios)",              "✅"),
    ("Conclusión con análisis basado en la práctica",             "✅"),
]

for item, estado in checklist:
    print(f"  {estado}  {item}")

print()
print(f"  Total de registros procesados: {df.count()}")
print(f"  Versión de Spark utilizada   : {spark.version}")
print()
print("  Archivo a entregar: Moreno_Jhan_Actividad_2.ipynb")
print("  Repositorio GitHub: github.com/[usuario]/Actividad_2")
print("="*65)

# Detener SparkSession al finalizar
spark.stop()
print("\n🛑  SparkSession cerrada correctamente")

  ✅  RESUMEN DE ACTIVIDAD COMPLETADA
  ✅  Diseño del esquema (StructType + DDL + diccionario)
  ✅  Configuración del entorno Spark evidenciada
  ✅  Dataset cargado desde CSV con esquema aplicado
  ✅  Tabla persistida en Parquet y vista SQL registrada
  ✅  DESCRIBE TABLE ejecutado en SQL
  ✅  printSchema() ejecutado en PySpark
  ✅  describe() / estadísticas en PySpark y SQL
  ✅  SELECT comparado en PySpark y SQL
  ✅  GROUP BY (Categoría y Estado) en PySpark y SQL
  ✅  Filtros / WHERE en PySpark y SQL
  ✅  COUNT(*), LIMIT y muestras ejecutados
  ✅  Tabla comparativa SQL vs Spark (9 criterios)
  ✅  Conclusión con análisis basado en la práctica

  Total de registros procesados: 20
  Versión de Spark utilizada   : 3.5.0

  Archivo a entregar: Moreno_Jhan_Actividad_2.ipynb
  Repositorio GitHub: github.com/[usuario]/Actividad_2

🛑  SparkSession cerrada correctamente
